##Downloading Dataset

In [ ]:
!pip install scikit-learn

In [ ]:
!pip install pandas numpy

In [ ]:
!wget 'https://zenodo.org/records/7119953/files/clue.zip?download=1' -O clue.zip

--2026-05-04 13:31:40--  https://zenodo.org/records/7119953/files/clue.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 137.138.153.219, 188.184.103.118, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635105552 (606M) [application/octet-stream]
Saving to: ‘clue.zip’

clue.zip            100%[===================>] 605.68M  29.7MB/s    in 20s     

2026-05-04 13:32:01 (29.7 MB/s) - ‘clue.zip’ saved [635105552/635105552]



In [ ]:
!unzip clue.zip

Archive:  clue.zip
  inflating: clue.json               


In [ ]:
!ls

clue.json  clue.zip  sample_data


Fixing Parameters: fixed UIDs + windows

In [ ]:
import pandas as pd

# Full dataset JSONL (as in your notebook)
FULL_JSONL = "clue.json"

# Keep SAME users across windows (your choices)
CONSISTENT_UID = "spectacular-copper-cheetah-postman"
INCONSISTENT_UID = "ready-silver-angelfish-quarryworker"
UIDS = {CONSISTENT_UID, INCONSISTENT_UID}

# Start date (use the same anchor as your 3-month experiment)
DATA_START = pd.Timestamp("2017-07-07", tz="UTC")

def window_end(start, months: int) -> pd.Timestamp:
    # inclusive end date
    return start + pd.DateOffset(months=months) - pd.Timedelta(days=1)

WINDOWS = {
    "3m":  (DATA_START, window_end(DATA_START, 3)),
    "6m":  (DATA_START, window_end(DATA_START, 6)),
    "12m": (DATA_START, window_end(DATA_START, 12)),
}

for k, (s, e) in WINDOWS.items():
    print(k, ":", s.date(), "→", e.date())

3m : 2017-07-07 → 2017-10-06
6m : 2017-07-07 → 2018-01-06
12m : 2017-07-07 → 2018-07-06


Extract window-specific JSONL for only those two UIDs

In [ ]:
import json
import pandas as pd

out_paths = {w: f"clue_uids_{w}.jsonl" for w in WINDOWS}
outs = {w: open(path, "w", encoding="utf-8") for w, path in out_paths.items()}

counts = {w: 0 for w in WINDOWS}
bad_json = 0
bad_time = 0

with open(FULL_JSONL, "r", encoding="utf-8") as fin:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            bad_json += 1
            continue

        uid = str(obj.get("uid", ""))
        if uid not in UIDS:
            continue

        t_raw = obj.get("time")
        if t_raw is None:
            bad_time += 1
            continue

        t = pd.to_datetime(t_raw, errors="coerce", utc=True)
        if pd.isna(t):
            bad_time += 1
            continue

        for w, (start, end) in WINDOWS.items():
            if start <= t <= end:
                outs[w].write(json.dumps(obj) + "\n")
                counts[w] += 1

for f in outs.values():
    f.close()

print("Done extracting.")
print("Counts per window:", counts)
print("bad_json:", bad_json, "bad_time:", bad_time)
print("Outputs:", out_paths)

Done extracting.
Counts per window: {'3m': 24088, '6m': 34121, '12m': 60745}
bad_json: 0 bad_time: 0
Outputs: {'3m': 'clue_uids_3m.jsonl', '6m': 'clue_uids_6m.jsonl', '12m': 'clue_uids_12m.jsonl'}


Build Daily features

In [ ]:
import pandas as pd
import numpy as np

FEATURES = [
    "events_total",
    "active_hours",
    "night_fraction",
    "unique_types",
    "file_events",
    "login_attempt",
    "login_successful",
    "login_success_rate",
    "unique_paths",
    "path_depth_mean",
    "unique_dir1",
    "unique_dir2",
    "path_reuse_ratio",
]

def build_daily_features_for_uid(jsonl_path: str, uid: str, out_csv: str, chunksize: int = 200_000):
    def path_depth(path: str) -> int:
        if not isinstance(path, str):
            return 0
        parts = [p for p in path.split("/") if p]
        return len(parts)

    def get_dir(path: str, idx: int):
        if not isinstance(path, str):
            return None
        parts = [p for p in path.split("/") if p]
        return parts[idx] if len(parts) > idx else None

    uid = str(uid)
    rows = []

    for chunk in pd.read_json(jsonl_path, lines=True, chunksize=chunksize):
        if "uid" not in chunk.columns or "time" not in chunk.columns:
            continue

        chunk["uid"] = chunk["uid"].astype(str)
        sub = chunk.loc[chunk["uid"] == uid].copy()
        if sub.empty:
            continue

        sub["time"] = pd.to_datetime(sub["time"], errors="coerce", utc=True)
        sub = sub.dropna(subset=["time"])

        sub["date"] = sub["time"].dt.date.astype(str)
        sub["hour"] = sub["time"].dt.hour

        if "params" in sub.columns:
            sub["params_path"] = sub["params"].apply(lambda x: x.get("path") if isinstance(x, dict) else None)
            sub["params_nkeys"] = sub["params"].apply(lambda x: len(x) if isinstance(x, dict) else 0)
        else:
            sub["params_path"] = None
            sub["params_nkeys"] = 0

        sub["path_depth"] = sub["params_path"].apply(path_depth)
        sub["dir1"] = sub["params_path"].apply(lambda p: get_dir(p, 0))
        sub["dir2"] = sub["params_path"].apply(lambda p: get_dir(p, 1))

        sub["is_file_accessed"] = (sub["type"] == "file_accessed").astype(int)
        sub["is_login_attempt"] = (sub["type"] == "login_attempt").astype(int)
        sub["is_login_successful"] = (sub["type"] == "login_successful").astype(int)

        rows.append(sub[[
            "date","hour","type","params_nkeys",
            "is_file_accessed","is_login_attempt","is_login_successful",
            "params_path","path_depth","dir1","dir2"
        ]])

    if not rows:
        raise RuntimeError(f"No rows found for uid={uid} in {jsonl_path}")

    user_df = pd.concat(rows, ignore_index=True)
    g = user_df.groupby("date", sort=True)

    daily = pd.DataFrame({
        "events_total": g.size(),
        "active_hours": g["hour"].nunique(),
        "first_hour": g["hour"].min(),
        "last_hour": g["hour"].max(),
        "night_events": g["hour"].apply(lambda s: ((s >= 0) & (s <= 5)).sum()),
        "unique_types": g["type"].nunique(),
        "params_key_count_mean": g["params_nkeys"].mean(),
        "file_events": g["is_file_accessed"].sum(),
        "login_attempt": g["is_login_attempt"].sum(),
        "login_successful": g["is_login_successful"].sum(),
        "unique_paths": g["params_path"].nunique(dropna=True),
        "path_depth_mean": g["path_depth"].mean(),
        "unique_dir1": g["dir1"].nunique(dropna=True),
        "unique_dir2": g["dir2"].nunique(dropna=True),
    }).reset_index().sort_values("date").reset_index(drop=True)

    eps = 1e-9
    daily["night_fraction"] = daily["night_events"] / (daily["events_total"] + eps)
    daily["login_success_rate"] = daily["login_successful"] / (daily["login_attempt"] + eps)
    daily["path_reuse_ratio"] = daily["file_events"] / (daily["unique_paths"] + eps)

    daily.to_csv(out_csv, index=False)
    return daily

Anomaly models

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

def global_baseline_anomaly(df: pd.DataFrame, features, thresh: str = "p95") -> pd.DataFrame:
    """
    Global baseline anomaly scoring.
    - Compute global percentile threshold per feature within the provided window df.
    - A day is "flagged" if any feature exceeds its threshold.
    - anomaly_score is sum over flagged features of (x/thresh - 1).
    """
    eps = 1e-9
    thresh = thresh.lower()
    if thresh not in ("p95", "p99"):
        raise ValueError("thresh must be 'p95' or 'p99'")

    q = 95 if thresh == "p95" else 99

    baseline = {}
    for f in features:
        s = df[f].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
        baseline[f] = float(np.nanpercentile(s, q)) if not s.empty else np.nan

    scores, n_flags, explanations = [], [], []

    for _, row in df.iterrows():
        contribs = []
        score = 0.0
        flags = 0

        for f in features:
            t = baseline.get(f, np.nan)
            if np.isnan(t):
                continue

            x = float(row[f])
            if x > t:
                r = x / (t + eps)
                c = r - 1.0
                score += c
                flags += 1
                contribs.append((f, x, t, r, c))

        contribs.sort(key=lambda z: z[4], reverse=True)
        top = contribs[:5]

        if top:
            expl = "; ".join([f"{f}={x:.3g} vs {thresh}={t:.3g} ({r:.2f}x)"
                              for (f, x, t, r, _) in top])
        else:
            expl = f"No features exceeded {thresh}"

        scores.append(score)
        n_flags.append(flags)
        explanations.append(expl)

    out = df.copy()
    out["baseline_thresh"] = thresh
    out["anomaly_score"] = scores
    out["num_flagged_features"] = n_flags
    out["top_contributors"] = explanations
    return out.sort_values(["anomaly_score", "num_flagged_features"], ascending=False)


def isolation_forest_anomaly(
    df: pd.DataFrame,
    features,
    contamination: float = 0.05,
    random_state: int = 42,
    n_estimators: int = 500,
) -> pd.DataFrame:
    """
    Isolation Forest anomaly scoring on daily features.
    Returns higher iso_anomaly_score = more anomalous.
    """
    X = df[features].replace([np.inf, -np.inf], np.nan)
    X = X.apply(lambda col: col.fillna(col.median()), axis=0)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = IsolationForest(
        n_estimators=n_estimators,
        contamination=contamination,
        random_state=random_state,
    )
    model.fit(X_scaled)

    pred = model.predict(X_scaled)  # 1=inlier, -1=outlier
    normality = model.decision_function(X_scaled)  # higher=more normal
    anomaly_score = -normality

    out = df[["date"]].copy()
    out["iso_pred"] = pred
    out["iso_anomaly_score"] = anomaly_score
    return out.sort_values("iso_anomaly_score", ascending=False)

Run for each window

In [ ]:
import pandas as pd

BASELINES = ["p95", "p99"]
WINDOWS_LIST = ["3m", "6m", "12m"]

def run_for_window(window_name: str):
    jsonl = f"clue_uids_{window_name}.jsonl"

    for uid in [CONSISTENT_UID, INCONSISTENT_UID]:
        # Daily features
        daily_csv = f"daily_features_{uid}_{window_name}.csv"
        daily = build_daily_features_for_uid(jsonl, uid, daily_csv)

        # Global baseline (both thresholds)
        for b in BASELINES:
            gb = global_baseline_anomaly(daily, FEATURES, thresh=b)
            gb.to_csv(f"anomaly_scores_global_baseline_{uid}_{window_name}_{b}.csv", index=False)

        # Isolation Forest
        iso = isolation_forest_anomaly(daily, FEATURES, contamination=0.05)
        iso.to_csv(f"isolation_forest_scores_{uid}_{window_name}.csv", index=False)

        print(f"Saved outputs for window={window_name}, uid={uid} | days={len(daily)}")

for w in WINDOWS_LIST:
    run_for_window(w)

print("Done running all windows.")

Saved outputs for window=3m, uid=spectacular-copper-cheetah-postman | days=90
Saved outputs for window=3m, uid=ready-silver-angelfish-quarryworker | days=84
Saved outputs for window=6m, uid=spectacular-copper-cheetah-postman | days=178
Saved outputs for window=6m, uid=ready-silver-angelfish-quarryworker | days=163
Saved outputs for window=12m, uid=spectacular-copper-cheetah-postman | days=350
Saved outputs for window=12m, uid=ready-silver-angelfish-quarryworker | days=334
Done running all windows.


Window and Baseline Effect summary

In [ ]:
import pandas as pd
import numpy as np

BASELINES = ["p95", "p99"]
WINDOWS_LIST = ["3m", "6m", "12m"]
UID_LIST = [CONSISTENT_UID, INCONSISTENT_UID]

def stability_metrics(daily: pd.DataFrame):
    x = daily["events_total"].astype(float).values
    mean = float(np.mean(x))
    std = float(np.std(x))
    cv = std / (mean + 1e-9)
    med = float(np.median(x))
    p95 = float(np.percentile(x, 95))
    burst = p95 / max(1.0, med)
    return mean, med, p95, cv, burst

rows = []

for uid in UID_LIST:
    for w in WINDOWS_LIST:
        daily = pd.read_csv(f"daily_features_{uid}_{w}.csv").sort_values("date")
        mean, med, p95, cv, burst = stability_metrics(daily)

        row = {
            "uid": uid,
            "window": w,
            "days_observed": int(len(daily)),
            "events_mean": mean,
            "events_median": med,
            "events_p95": p95,
            "cv_events": cv,
            "burst_ratio_p95_over_median": burst,
        }

        for b in BASELINES:
            gb = pd.read_csv(f"anomaly_scores_global_baseline_{uid}_{w}_{b}.csv")
            row[f"baseline_{b}_frac_days_flagged"] = float((gb["num_flagged_features"] > 0).mean())
            row[f"baseline_{b}_max_anomaly_score"] = float(gb["anomaly_score"].max()) if len(gb) else np.nan
            row[f"baseline_{b}_median_anomaly_score"] = float(gb["anomaly_score"].median()) if len(gb) else np.nan

        iso = pd.read_csv(f"isolation_forest_scores_{uid}_{w}.csv")
        row["isoforest_frac_outliers"] = float((iso["iso_pred"] == -1).mean())
        row["isoforest_max_anomaly_score"] = float(iso["iso_anomaly_score"].max()) if len(iso) else np.nan
        row["isoforest_median_anomaly_score"] = float(iso["iso_anomaly_score"].median()) if len(iso) else np.nan

        rows.append(row)

summary = pd.DataFrame(rows).sort_values(["uid", "window"])
display(summary)

summary.to_csv("window_effect_summary.csv", index=False)
print("Saved: window_effect_summary.csv")

,uid,window,days_observed,events_mean,events_median,events_p95,cv_events,burst_ratio_p95_over_median,baseline_p95_frac_days_flagged,baseline_p95_max_anomaly_score,baseline_p95_median_anomaly_score,baseline_p99_frac_days_flagged,baseline_p99_max_anomaly_score,baseline_p99_median_anomaly_score,isoforest_frac_outliers,isoforest_max_anomaly_score,isoforest_median_anomaly_score
5,ready-silver-angelfish-quarryworker,12m,334,92.101796,31.0,328.35,4.669571,10.591935,0.287425,189.258885,0.0,0.092814,49.380826,0.0,0.050898,0.290577,-0.139595
3,ready-silver-angelfish-quarryworker,3m,84,194.988095,55.0,487.75,4.198049,8.868182,0.309524,73.505176,0.0,0.083333,13.141112,0.0,0.059524,0.242617,-0.136024
4,ready-silver-angelfish-quarryworker,6m,163,121.441718,37.0,364.10,4.892764,9.840541,0.319018,114.609095,0.0,0.079755,49.496216,0.0,0.055215,0.275646,-0.147007
2,spectacular-copper-cheetah-postman,12m,350,85.665714,68.0,123.55,1.687522,1.816912,0.260000,109.413833,0.0,0.062857,8.998055,0.0,0.051429,0.243894,-0.129163
0,spectacular-copper-cheetah-postman,3m,90,85.655556,73.5,151.10,0.876313,2.055782,0.244444,22.694392,0.0,0.077778,1.996661,0.0,0.055556,0.177655,-0.131405
1,spectacular-copper-cheetah-postman,6m,178,80.483146,76.0,138.30,0.754835,1.819737,0.269663,25.357498,0.0,0.067416,2.509573,0.0,0.050562,0.186128,-0.140955


Saved: window_effect_summary.csv


Percentage of flagged days

In [ ]:
import pandas as pd
import numpy as np

BASELINES = ["p95", "p99"]
WINDOWS_LIST = ["3m", "6m", "12m"]
UID_LIST = [CONSISTENT_UID, INCONSISTENT_UID]

rows = []

for uid in UID_LIST:
    for w in WINDOWS_LIST:
        daily = pd.read_csv(f"daily_features_{uid}_{w}.csv")
        days_total = len(daily)

        for b in BASELINES:
            gb = pd.read_csv(f"anomaly_scores_global_baseline_{uid}_{w}_{b}.csv")

            flagged_mask = gb["num_flagged_features"] > 0
            days_flagged = int(flagged_mask.sum())
            flagged_pct = float(days_flagged / days_total) if days_total else np.nan

            rows.append({
                "uid": uid,
                "window": w,
                "baseline": b,
                "days_total": int(days_total),
                "days_flagged": days_flagged,
                "flagged_pct": flagged_pct,
                "avg_num_flagged_features_on_flagged_days": float(gb.loc[flagged_mask, "num_flagged_features"].mean()) if days_flagged else 0.0,
            })

flagged_summary = pd.DataFrame(rows).sort_values(["uid", "window", "baseline"])
display(flagged_summary)

flagged_summary.to_csv("flagged_days_percentage.csv", index=False)
print("Saved: flagged_days_percentage.csv")

,uid,window,baseline,days_total,days_flagged,flagged_pct,avg_num_flagged_features_on_flagged_days
10,ready-silver-angelfish-quarryworker,12m,p95,334,96,0.287425,2.114583
11,ready-silver-angelfish-quarryworker,12m,p99,334,31,0.092814,1.677419
6,ready-silver-angelfish-quarryworker,3m,p95,84,26,0.309524,2.153846
7,ready-silver-angelfish-quarryworker,3m,p99,84,7,0.083333,1.857143
8,ready-silver-angelfish-quarryworker,6m,p95,163,52,0.319018,2.019231
9,ready-silver-angelfish-quarryworker,6m,p99,163,13,0.079755,1.923077
4,spectacular-copper-cheetah-postman,12m,p95,350,91,0.260000,2.076923
5,spectacular-copper-cheetah-postman,12m,p99,350,22,0.062857,2.045455
0,spectacular-copper-cheetah-postman,3m,p95,90,22,0.244444,2.227273
1,spectacular-copper-cheetah-postman,3m,p99,90,7,0.077778,1.714286


Saved: flagged_days_percentage.csv


Overlap

In [ ]:
import pandas as pd
from IPython.display import display

PAIRS = [("3m", "6m"), ("3m", "12m"), ("6m", "12m")]
BASELINES = ["p95", "p99"]
UID_LIST = [CONSISTENT_UID, INCONSISTENT_UID]

def topk_dates_global(uid: str, window: str, baseline: str, k_top: int):
    path = f"anomaly_scores_global_baseline_{uid}_{window}_{baseline}.csv"
    df = pd.read_csv(path)
    df = df.sort_values(["anomaly_score", "num_flagged_features"], ascending=False)
    return list(df["date"].astype(str).head(k_top))

def topk_dates_iso(uid: str, window: str, k_top: int):
    path = f"isolation_forest_scores_{uid}_{window}.csv"
    df = pd.read_csv(path)
    df = df.sort_values("iso_anomaly_score", ascending=False)
    return list(df["date"].astype(str).head(k_top))

def overlap_stats(A, B):
    Aset, Bset = set(A), set(B)
    inter = Aset & Bset
    union = Aset | Bset
    return {
        "overlap": len(inter),
        "jaccard": (len(inter) / len(union)) if union else 0.0,
        "overlap_frac_of_A": (len(inter) / len(Aset)) if Aset else 0.0,
        "overlap_frac_of_B": (len(inter) / len(Bset)) if Bset else 0.0,
        "overlap_dates": sorted(inter),
    }

def run_overlap(uid: str, k_top: int = 10):
    rows = []

    for baseline in BASELINES:
        for w1, w2 in PAIRS:
            A = topk_dates_global(uid, w1, baseline, k_top)
            B = topk_dates_global(uid, w2, baseline, k_top)
            st = overlap_stats(A, B)
            rows.append({
                "uid": uid,
                "method": f"global_baseline_{baseline}",
                "k": k_top,
                "window_1": w1,
                "window_2": w2,
                "overlap": st["overlap"],
                "jaccard": st["jaccard"],
                "overlap_frac_of_A": st["overlap_frac_of_A"],
                "overlap_frac_of_B": st["overlap_frac_of_B"],
                "overlap_dates": "; ".join(map(str, st["overlap_dates"])),
            })

    for w1, w2 in PAIRS:
        A = topk_dates_iso(uid, w1, k_top)
        B = topk_dates_iso(uid, w2, k_top)
        st = overlap_stats(A, B)
        rows.append({
            "uid": uid,
            "method": "isolation_forest",
            "k": k_top,
            "window_1": w1,
            "window_2": w2,
            "overlap": st["overlap"],
            "jaccard": st["jaccard"],
            "overlap_frac_of_A": st["overlap_frac_of_A"],
            "overlap_frac_of_B": st["overlap_frac_of_B"],
            "overlap_dates": "; ".join(map(str, st["overlap_dates"])),
        })

    return pd.DataFrame(rows)

K = 10

overlap_all = pd.concat([run_overlap(uid, k_top=K) for uid in UID_LIST], ignore_index=True)
overlap_all = overlap_all.sort_values(["uid", "method", "window_1", "window_2"])

display(overlap_all)
overlap_all.to_csv(f"top{K}_overlap_across_windows.csv", index=False)
print(f"Saved: top{K}_overlap_across_windows.csv")

,uid,method,k,window_1,window_2,overlap,jaccard,overlap_frac_of_A,overlap_frac_of_B,overlap_dates
10,ready-silver-angelfish-quarryworker,global_baseline_p95,10,3m,12m,5,0.333333,0.5,0.5,2017-07-11; 2017-07-17; 2017-07-21; 2017-08-18...
9,ready-silver-angelfish-quarryworker,global_baseline_p95,10,3m,6m,6,0.428571,0.6,0.6,2017-07-11; 2017-07-17; 2017-07-21; 2017-07-23...
11,ready-silver-angelfish-quarryworker,global_baseline_p95,10,6m,12m,7,0.538462,0.7,0.7,2017-07-11; 2017-07-17; 2017-07-21; 2017-08-18...
13,ready-silver-angelfish-quarryworker,global_baseline_p99,10,3m,12m,2,0.111111,0.2,0.2,2017-07-11; 2017-08-18
12,ready-silver-angelfish-quarryworker,global_baseline_p99,10,3m,6m,5,0.333333,0.5,0.5,2017-07-11; 2017-07-14; 2017-07-21; 2017-08-18...
14,ready-silver-angelfish-quarryworker,global_baseline_p99,10,6m,12m,5,0.333333,0.5,0.5,2017-07-11; 2017-07-23; 2017-08-18; 2017-12-19...
16,ready-silver-angelfish-quarryworker,isolation_forest,10,3m,12m,6,0.428571,0.6,0.6,2017-07-11; 2017-07-17; 2017-07-21; 2017-07-23...
15,ready-silver-angelfish-quarryworker,isolation_forest,10,3m,6m,8,0.666667,0.8,0.8,2017-07-10; 2017-07-11; 2017-07-17; 2017-07-21...
17,ready-silver-angelfish-quarryworker,isolation_forest,10,6m,12m,7,0.538462,0.7,0.7,2017-07-11; 2017-07-17; 2017-07-21; 2017-07-23...
1,spectacular-copper-cheetah-postman,global_baseline_p95,10,3m,12m,3,0.176471,0.3,0.3,2017-07-17; 2017-07-28; 2017-08-24


Saved: top10_overlap_across_windows.csv


per-window thresholds (p95/p99) for each feature

In [ ]:
import pandas as pd
import numpy as np

BASELINES = ["p95", "p99"]
WINDOWS_LIST = ["3m", "6m", "12m"]
UID_LIST = [CONSISTENT_UID, INCONSISTENT_UID]

rows = []

for uid in UID_LIST:
    for w in WINDOWS_LIST:
        daily = pd.read_csv(f"daily_features_{uid}_{w}.csv")

        for feat in FEATURES:
            s = pd.to_numeric(daily[feat], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

            if len(s) == 0:
                p95 = np.nan
                p99 = np.nan
            else:
                p95 = float(np.nanpercentile(s, 95))
                p99 = float(np.nanpercentile(s, 99))

            rows.append({
                "uid": uid,
                "window": w,
                "feature": feat,
                "p95_threshold": p95,
                "p99_threshold": p99,
                "n_days": int(len(daily)),
                "n_nonnull": int(s.shape[0]),
            })

thresholds = pd.DataFrame(rows).sort_values(["uid", "feature", "window"])
display(thresholds)

thresholds.to_csv("baseline_thresholds_by_window.csv", index=False)
print("Saved: baseline_thresholds_by_window.csv")

,uid,window,feature,p95_threshold,p99_threshold,n_days,n_nonnull
66,ready-silver-angelfish-quarryworker,12m,active_hours,15.00,17.67,334,334
40,ready-silver-angelfish-quarryworker,3m,active_hours,16.00,19.17,84,84
53,ready-silver-angelfish-quarryworker,6m,active_hours,16.00,18.38,163,163
65,ready-silver-angelfish-quarryworker,12m,events_total,328.35,828.37,334,334
39,ready-silver-angelfish-quarryworker,3m,events_total,487.75,2006.33,84,84
...,...,...,...,...,...,...,...
8,spectacular-copper-cheetah-postman,3m,unique_paths,29.65,231.96,90,90
21,spectacular-copper-cheetah-postman,6m,unique_paths,25.45,236.28,178,178
29,spectacular-copper-cheetah-postman,12m,unique_types,5.00,6.00,350,350
3,spectacular-copper-cheetah-postman,3m,unique_types,5.00,6.22,90,90


Saved: baseline_thresholds_by_window.csv


In [ ]:
from google.colab import files
files.download('baseline_thresholds_by_window.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>